In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

plt.style.use('ggplot')
sns.set_palette("magma")
print("Environment Ready.")

In [ ]:
try:
    data_path = '../data/processed/gaming_data_cleaned.csv'
    df = pd.read_csv(data_path)
    print(f"Dataset loaded. Total Records: {len(df)}")
except FileNotFoundError as e:
    print(f"Error: {e}")
    raise

In [ ]:
sys.path.insert(0, '../src')
from feature_engineering import COLUMN_MAPPING

df = df.rename(columns=COLUMN_MAPPING)
print("Columns synchronized.")
df.head()

In [ ]:
# Checking the distribution health of engagement metrics
df[['session_duration_hr', 'weekly_frequency', 'milestones_reached']].describe()

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df['session_duration_hr'], bins=30, kde=True)
plt.title('Distribution of User Session Durations')
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='weekly_frequency')
plt.title('Distribution of Weekly Login Frequency')
plt.show()

In [ ]:
Q1 = df['session_duration_hr'].quantile(0.25)
Q3 = df['session_duration_hr'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR

outliers = df[df['session_duration_hr'] > upper_bound]
print(f"Outliers detected: {len(outliers)}")

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='session_duration_hr', y='milestones_reached', alpha=0.3)
plt.title('Correlation: Play Time vs. Progression')
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[['session_duration_hr', 'weekly_frequency', 'milestones_reached']].corr(), annot=True, cmap='coolwarm')
plt.title('Engagement Feature Heatmap')
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[['session_duration_hr', 'weekly_frequency', 'milestones_reached']])
print("Scaling Complete.")

In [ ]:
from sklearn.cluster import KMeans
wcss = []
for i in range(1, 11):
    km = KMeans(n_clusters=i, init='k-means++', random_state=42)
    km.fit(X_scaled)
    wcss.append(km.inertia_)

plt.plot(range(1, 11), wcss, marker='o')
plt.title('Elbow Method Optimization')
plt.show()


In [ ]:
np.save('../data/processed/scaled_gaming_data.npy', X_scaled)
print(f"Scaled data shape {X_scaled.shape} saved for downstream modeling.")
print(f"File location: ../data/processed/scaled_gaming_data.npy")